In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spr-2026-mammography-report-classification/submission.csv
/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv
/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE  # 关键：过采样
from imblearn.pipeline import Pipeline as ImbPipeline  # 可选，用于交叉验证
import nltk
from nltk.corpus import stopwords
import re
import string
# 读取数据（路径请按实际情况修改）
train = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/submission.csv")
# ---------- 修复预处理函数 ----------
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()                             # 1. 转小写
    text = text.translate(str.maketrans('', '', string.punctuation))  # 2. 去标点
    text = re.sub(r'\d+', '', text)                 # 3. 去数字
    text = re.sub(r'<.*?>', '', text)               # 4. 去特殊标记
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)                         # 5. 修复：用空格连接，不是 "".join
# 处理训练集和测试集
train['clean_report'] = train['report'].apply(preprocess_text)
test['clean_report'] = test['report'].apply(preprocess_text)
# ---------- TF-IDF 向量化 ----------
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train['clean_report'])
X_test_tfidf = tfidf.transform(test['clean_report'])
y_train = train['target']   # BI-RADS 0~6
# ---------- 先划分训练集和验证集（stratify保证验证集也保留原始分布）----------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tfidf, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)
# ========== 1. 逻辑回归 + SMOTE ==========
print("=== 逻辑回归 + SMOTE ===")
# 只在训练集 X_tr 上做 SMOTE（不要动验证集）
smote = SMOTE(random_state=42, k_neighbors=3)   # k_neighbors 可以调，少数类太少时减小
X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr, y_tr)
# 训练逻辑回归（class_weight 可以继续用 balanced，或者用 None 因为已经过采样）
lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr.fit(X_tr_resampled, y_tr_resampled)
y_pred_val = lr.predict(X_val)
print("逻辑回归 F1 Score (macro):", f1_score(y_val, y_pred_val, average='macro'))
print(classification_report(y_val, y_pred_val))
y_test_pred = lr.predict(X_test_tfidf)
pred_dict = dict(zip(test['ID'], y_test_pred))
submission['target'] = submission['ID'].map(pred_dict)
 # 3. 保存提交文件
submission.to_csv("submission.csv", index=False)
print("submission.csv 已生成！")

[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


=== 逻辑回归 + SMOTE ===
逻辑回归 F1 Score (macro): 0.6955825279645349
              precision    recall  f1-score   support

           0       0.65      0.74      0.69       122
           1       0.90      0.96      0.93       138
           2       0.99      0.95      0.97      3194
           3       0.43      0.69      0.53       143
           4       0.53      0.65      0.58        43
           5       0.33      0.17      0.22         6
           6       1.00      0.89      0.94         9

    accuracy                           0.93      3655
   macro avg       0.69      0.72      0.70      3655
weighted avg       0.95      0.93      0.94      3655

submission.csv 已生成！


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import nltk
from nltk.corpus import stopwords
import re
import string
# -------------------------- 1. 固定所有随机种子，消除波动 --------------------------
SEED = 42
np.random.seed(SEED)
# 停用词（去掉下载语句，避免Kaggle报错）
stop_words = set(stopwords.words('portuguese'))
# -------------------------- 2. 读取数据 + 极少量类手动增强 --------------------------
train = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/submission.csv")
# 【安全优化1】手动复制极少量类（5、6类）样本，提升训练权重
minority_classes = [5, 6]
train_minority = train[train['target'].isin(minority_classes)]
train = pd.concat([train, train_minority, train_minority], ignore_index=True)
# -------------------------- 3. 文本预处理（稳定基线版） --------------------------
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)
train['clean_report'] = train['report'].apply(preprocess_text)
test['clean_report'] = test['report'].apply(preprocess_text)
# -------------------------- 4. TF-IDF 微调版 --------------------------
# 【安全优化2】max_features从5000提升到6000，过滤低频噪声词
tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    min_df=2
)
X_train_tfidf = tfidf.fit_transform(train['clean_report'])
X_test_tfidf = tfidf.transform(test['clean_report'])
y_train = train['target']
# -------------------------- 5. 划分训练/验证集（固定随机种子） --------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tfidf, y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)
# -------------------------- 6. SMOTE + 逻辑回归（全固定随机种子） --------------------------
print("=== 逻辑回归 + SMOTE ===")
smote = SMOTE(random_state=SEED, k_neighbors=3)
X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr, y_tr)
lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=1,
    random_state=SEED
)
lr.fit(X_tr_resampled, y_tr_resampled)
# -------------------------- 7. 验证集预测与评估 --------------------------
y_pred_val = lr.predict(X_val)
print("逻辑回归 F1 Score (macro):", f1_score(y_val, y_pred_val, average='macro'))
print(classification_report(y_val, y_pred_val, zero_division=0))
# -------------------------- 8. 测试集预测与生成提交文件 --------------------------
y_test_pred = lr.predict(X_test_tfidf)
pred_dict = dict(zip(test['ID'], y_test_pred))
submission['target'] = submission['ID'].map(pred_dict)
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv 已生成！")

=== 逻辑回归 + SMOTE ===
逻辑回归 F1 Score (macro): 0.8181266946377784
              precision    recall  f1-score   support

           0       0.66      0.74      0.70       122
           1       0.90      0.96      0.93       139
           2       0.99      0.96      0.97      3194
           3       0.44      0.72      0.55       142
           4       0.59      0.63      0.61        43
           5       0.94      1.00      0.97        17
           6       1.00      1.00      1.00        27

    accuracy                           0.94      3684
   macro avg       0.79      0.86      0.82      3684
weighted avg       0.95      0.94      0.94      3684

✅ submission.csv 已生成！


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
import re
import string

# 1. 固定随机种子
SEED = 42
np.random.seed(SEED)

# 2. 初始化工具
stop_words = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()

# 3. 读取数据 + 增强所有少数类
train = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/spr-2026-mammography-report-classification/submission.csv")

minority_classes = [3, 4, 5, 6]
train_minority = train[train['target'].isin(minority_classes)]
train = pd.concat([train, train_minority, train_minority], ignore_index=True)

# 4. 升级版预处理（词干提取+过滤短词）
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens if len(word) > 2 and word not in stop_words]
    return ' '.join(tokens)

train['clean_report'] = train['report'].apply(preprocess_text)
test['clean_report'] = test['report'].apply(preprocess_text)

# 5. 升级版TF-IDF
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

X_train_tfidf = tfidf.fit_transform(train['clean_report'])
X_test_tfidf = tfidf.transform(test['clean_report'])
y_train = train['target']

# 6. 划分训练集
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tfidf, y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)

# 7. SMOTE + 逻辑回归
smote = SMOTE(random_state=SEED, k_neighbors=3)
X_tr_resampled, y_tr_resampled = smote.fit_resample(X_tr, y_tr)

lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=0.5,
    random_state=SEED,
    n_jobs=-1
)
lr.fit(X_tr_resampled, y_tr_resampled)

# 8. 验证集评估
y_pred_val = lr.predict(X_val)
print("逻辑回归 F1 Score (macro):", f1_score(y_val, y_pred_val, average='macro'))
print(classification_report(y_val, y_pred_val, zero_division=0))

# 9. 生成提交文件
y_test_pred = lr.predict(X_test_tfidf)
pred_dict = dict(zip(test['ID'], y_test_pred))
submission['target'] = submission['ID'].map(pred_dict)
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv 已生成！")

逻辑回归 F1 Score (macro): 0.8921250489731022
              precision    recall  f1-score   support

           0       0.71      0.76      0.74       122
           1       0.85      0.96      0.90       139
           2       1.00      0.94      0.97      3194
           3       0.70      0.91      0.79       428
           4       0.84      0.96      0.90       128
           5       0.94      1.00      0.97        17
           6       0.96      1.00      0.98        27

    accuracy                           0.93      4055
   macro avg       0.86      0.93      0.89      4055
weighted avg       0.94      0.93      0.94      4055

✅ submission.csv 已生成！
